In [15]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np

ol = Overlay("/root/jupyter_notebooks/rhd_spi_0807/test_ila_data_o/rhd_spi_ip_0806_wrapper.bit")

In [16]:
ol?

In [17]:
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
IE_IS_REG   = 0x14  # 中斷控制暫存器
DATA_RO_REG = 0x18  # data valid flag
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [18]:
Valid = mmio.read(DATA_RO_REG)
print(f"SPI valid: 0x{Valid:08x}")
# print(f"ENABLE: {status & 0x1}")
# print(f"STATUS: {(status >> 1) & 0x1}")
# print(f"PENDING: {(status >> 1) & 0x1}")


SPI valid: 0x00000000


In [10]:
def read_once():
#     while mmio.read(STATUS) & 1:   # 等 busy=0
#         pass
    return mmio.read(RX_DATA_REG) & 0xFFFF

mmio.write(PHA_SEL_REG, 0x01)
mmio.write(CONTROL_REG, 0x01)          # start_pulse
results = {}

for i in range(20):
    while (Valid):
        results[i] = read_once()
print (results)
for i,val in results.items():
    print(f"data {i:2d}: 0x{val:04X}")

{}


In [14]:
results_2 = {}
for i in range(20):
    results_2[i] = mmio.read(DATA_RO_REG) & 0xFFFF
    
for i,val in results_2.items():
    print(f"data {i:2d}: 0x{val:04X}")

data  0: 0x0000
data  1: 0x0000
data  2: 0x0000
data  3: 0x0000
data  4: 0x0000
data  5: 0x0000
data  6: 0x0000
data  7: 0x0000
data  8: 0x0000
data  9: 0x0000
data 10: 0x0000
data 11: 0x0000
data 12: 0x0000
data 13: 0x0000
data 14: 0x0000
data 15: 0x0000
data 16: 0x0000
data 17: 0x0000
data 18: 0x0000
data 19: 0x0000


In [22]:
# reg5 現在用來interrupt
BASE   = 0xA0000000
RANGE  = 0x10000
# USER_REG = 0x14                 # slv_reg5

mmio = MMIO(BASE, RANGE)

patterns = [0xCAFEBABE, 0x0BAD_F00D, 0x55AA55AA]

for p in patterns:
    mmio.write(USER_REG, p)
    r = mmio.read(USER_REG)
    print(f"Write 0x{p:08X}  → Read 0x{r:08X} "
          f"{'OK' if r==p else 'FAIL'}")



NameError: name 'USER_REG' is not defined

In [19]:
mmio.write(PHA_SEL_REG, 0x2)
mmio.write(CONTROL_REG, 0x1)

In [4]:
import time
import threading

class SPIDriver:
    def __init__(self, overlay_path, base_addr):
        self.overlay = Overlay(overlay_path)
        self.mmio = MMIO(base_addr, 0x10000)
        
        # 暫存器偏移
        self.CTRL_REG = 0x00     # start_pulse[0], stop_pulse[1]
        self.PHASE_REG = 0x04    # phase_select[3:0]
        self.TX_DATA_REG = 0x08  # tx_data[15:0]
        self.RX_DATA_REG = 0x0C  # rx_data[15:0] (RO，讀取時清除中斷)
        self.STATUS_REG = 0x10   # busy[0], finish[1], data_valid[2]
        self.IE_IS_REG = 0x14    # 中斷控制暫存器
        
        # 中斷位元遮罩
        self.INT_ENABLE_MASK = 0x1        # bit[0]  = interrupt_enable
        self.INT_STATUS_MASK = 0x100      # bit[8]  = interrupt_status  
        self.INT_PENDING_MASK = 0x10000   # bit[16] = global_interrupt_pending
        
        self.is_running = False
        self.data_buffer = []

    def enable_interrupt(self):
        """啟用中斷 - 設置 bit[0] = 1"""
        self.mmio.write(self.IE_IS_REG, self.INT_ENABLE_MASK)
        print("Interrupt enabled via MMIO")
        
        # 驗證設置
        ie_is = self.mmio.read(self.IE_IS_REG)
        enabled = bool(ie_is & self.INT_ENABLE_MASK)
        print(f"Interrupt enable verified: {enabled}")

    def disable_interrupt(self):
        """停用中斷 - 設置 bit[0] = 0"""
        self.mmio.write(self.IE_IS_REG, 0x0)
        print("Interrupt disabled via MMIO")

    def check_interrupt_status(self):
        """檢查中斷狀態 - 純 MMIO 讀取"""
        ie_is = self.mmio.read(self.IE_IS_REG)
        
        interrupt_enable = bool(ie_is & self.INT_ENABLE_MASK)     # bit[0]
        interrupt_status = bool(ie_is & self.INT_STATUS_MASK)     # bit[8]
        global_pending = bool(ie_is & self.INT_PENDING_MASK)      # bit[16]
        
        return {
            'raw_value': ie_is,
            'interrupt_enable': interrupt_enable,
            'interrupt_status': interrupt_status,
            'global_pending': global_pending,
            'has_interrupt': interrupt_status or global_pending  # 任一為真表示有中斷
        }

    def wait_for_interrupt_mmio(self, timeout=5.0):
        """純 MMIO 方式等待中斷"""
        print("=== Waiting for interrupt via MMIO polling ===")
        
        start_time = time.time()
        
        while (time.time() - start_time) < timeout:
            # 檢查中斷狀態
            int_info = self.check_interrupt_status()
            
            if int_info['has_interrupt']:
                print(f"Interrupt detected! IE/IS = 0x{int_info['raw_value']:08X}")
                print(f"  interrupt_status: {int_info['interrupt_status']}")
                print(f"  global_pending:   {int_info['global_pending']}")
                
                # 讀取資料 (會自動清除中斷狀態)
                rx_data = self.mmio.read(self.RX_DATA_REG) & 0xFFFF
                print(f"  RX Data: 0x{rx_data:04X}")
                
                # 驗證中斷狀態被清除
                int_info_after = self.check_interrupt_status()
                print(f"  After read IE/IS = 0x{int_info_after['raw_value']:08X}")
                print(f"  Interrupt cleared: {not int_info_after['has_interrupt']}")
                
                return rx_data
            
            time.sleep(0.001)  # 1ms 輪詢間隔
        
        print("Timeout waiting for interrupt")
        return None

    def start_continuous_mmio_polling(self, phase_select=3, tx_data=0xFF00, poll_interval=0.005):
        """使用純 MMIO 輪詢方式進行連續採集"""
        print("=== Starting Continuous MMIO Polling ===")
        
        # 設定 SPI 參數
        self.mmio.write(self.PHASE_REG, phase_select)
        self.mmio.write(self.TX_DATA_REG, tx_data)
        print(f"SPI configured: phase={phase_select}, tx_data=0x{tx_data:04X}")
        
        # 啟用中斷
        self.enable_interrupt()
        
        # 發送 start pulse (連續傳輸)
        self.mmio.write(self.CTRL_REG, 0x1)  # start_pulse = 1
        print("SPI continuous transmission started")
        
        # 開始輪詢執行緒
        self.is_running = True
        self.polling_thread = threading.Thread(target=self._mmio_polling_worker, args=(poll_interval,))
        self.polling_thread.start()
        
        print(f"MMIO polling started (interval={poll_interval*1000:.1f}ms)")

    def _mmio_polling_worker(self, poll_interval):
        """MMIO 輪詢工作執行緒"""
        while self.is_running:
            try:
                # 檢查中斷狀態 - 純 MMIO 讀取
                int_info = self.check_interrupt_status()
                
                if int_info['has_interrupt']:
                    # 讀取資料 (會自動清除中斷狀態)
                    rx_data = self.mmio.read(self.RX_DATA_REG) & 0xFFFF
                    timestamp = time.time()
                    
                    # 儲存資料
                    self.data_buffer.append({
                        'data': rx_data,
                        'timestamp': timestamp,
                        'ie_is_before': int_info['raw_value']
                    })
                    
                    print(f"[{len(self.data_buffer):04d}] RX: 0x{rx_data:04X}, IE/IS: 0x{int_info['raw_value']:08X}")
                    
                    # 可選：檢查中斷是否被清除
                    if len(self.data_buffer) % 10 == 0:  # 每10筆資料檢查一次
                        int_info_after = self.check_interrupt_status()
                        cleared = not int_info_after['has_interrupt']
                        print(f"    Interrupt auto-cleared: {cleared}")
                
                time.sleep(poll_interval)
                
            except Exception as e:
                print(f"MMIO polling error: {e}")
                break

    def stop_acquisition(self):
        """停止採集"""
        print("Stopping acquisition...")
        
        # 停止輪詢
        self.is_running = False
        if hasattr(self, 'polling_thread'):
            self.polling_thread.join()
        
        # 發送 stop pulse
        self.mmio.write(self.CTRL_REG, 0x2)  # stop_pulse = 1
        
        # 停用中斷
        self.disable_interrupt()
        
        print(f"Acquisition stopped. Total data: {len(self.data_buffer)} samples")

    def test_interrupt_mechanism(self):
        """測試中斷機制"""
        print("=== Testing Interrupt Mechanism ===")
        
        # 1. 測試中斷使能
        print("1. Testing interrupt enable/disable...")
        self.enable_interrupt()
        int_info = self.check_interrupt_status()
        print(f"   After enable: {int_info}")
        
        self.disable_interrupt()
        int_info = self.check_interrupt_status()
        print(f"   After disable: {int_info}")
        
        # 2. 測試單次傳輸的中斷
        print("2. Testing single transmission interrupt...")
        self.mmio.write(self.PHASE_REG, 3)
        self.mmio.write(self.TX_DATA_REG, 0xFF00)
        
        self.enable_interrupt()
        self.mmio.write(self.CTRL_REG, 0x1)  # start_pulse
        
        # 等待中斷
        data = self.wait_for_interrupt_mmio(timeout=2.0)
        if data is not None:
            print(f"   ✓ Interrupt mechanism working! Received: 0x{data:04X}")
        else:
            print(f"   ✗ No interrupt detected")
        
        self.disable_interrupt()

In [5]:
try:
    spi = SPIDriver(
        overlay_path="/root/jupyter_notebooks/rhd_spi_0807/test_ila_data_o/rhd_spi_ip_0806_wrapper.bit",
        base_addr=0xA0000000
    )
    
    # 測試中斷機制
    spi.test_interrupt_mechanism()
    
    print("\n" + "="*50)
    
    # 連續採集測試
    spi.start_continuous_mmio_polling(phase_select=1, tx_data=0xFF00, poll_interval=0.000001)  # 10ms 輪詢
    
    # 運行 15 秒
    time.sleep(0.001)
    
    # 停止採集
    spi.stop_acquisition()
    
    # 檢視結果
    if spi.data_buffer:
        print("\nLatest 5 samples:")
        for i, sample in enumerate(spi.data_buffer[-5:]):
            print(f"  {i+1}: 0x{sample['data']:04X} @ {sample['timestamp']:.3f}s, IE/IS=0x{sample['ie_is_before']:08X}")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

=== Testing Interrupt Mechanism ===
1. Testing interrupt enable/disable...
Interrupt enabled via MMIO
Interrupt enable verified: True
   After enable: {'raw_value': 1, 'interrupt_enable': True, 'interrupt_status': False, 'global_pending': False, 'has_interrupt': False}
Interrupt disabled via MMIO
   After disable: {'raw_value': 0, 'interrupt_enable': False, 'interrupt_status': False, 'global_pending': False, 'has_interrupt': False}
2. Testing single transmission interrupt...
Interrupt enabled via MMIO
Interrupt enable verified: True
=== Waiting for interrupt via MMIO polling ===
Timeout waiting for interrupt
   ✗ No interrupt detected
Interrupt disabled via MMIO

=== Starting Continuous MMIO Polling ===
SPI configured: phase=1, tx_data=0xFF00
Interrupt enabled via MMIO
Interrupt enable verified: True
SPI continuous transmission started
MMIO polling started (interval=0.0ms)
Stopping acquisition...
Interrupt disabled via MMIO
Acquisition stopped. Total data: 0 samples


In [19]:
data_list = {}
# 輪詢方式 - 持續檢查暫存器位元
for i in range(16):
    ie_is = mmio.read(0x14)  # 讀取 IE/IS 暫存器
    interrupt_status = bool(ie_is & 0x100)  # 檢查 bit[8]
    global_pending = bool(ie_is & 0x10000)  # 檢查 bit[16]
    
    if interrupt_status or global_pending:
        # 有中斷事件，處理資料
        data = mmio.read(0x0C)  # 讀取會自動清除中斷狀態
        data_list[i] = get_latest_data(1)
    
    time.sleep(0.001)  # 1ms 輪詢間隔
print(f"RX = 0x{data_list}")

RX = 0x{}


In [13]:
def read_once():
    mmio.write(CTRL, 0x1)          # start_pulse
#     while mmio.read(STATUS) & 1:   # 等 busy=0
#         pass
    return mmio.read(RX_DATA) & 0xFFFF

results = {}
for ph in range(16):               # 0‥15 全掃
    mmio.write(PHASE_SEL, ph)
    results[ph] = read_once()

for ph,val in results.items():
    print(f"phase {ph:2d}: 0x{val:04X}")

phase  0: 0x0000
phase  1: 0x0000
phase  2: 0x0000
phase  3: 0x0000
phase  4: 0x0000
phase  5: 0x0000
phase  6: 0x0000
phase  7: 0x0000
phase  8: 0x0000
phase  9: 0x0000
phase 10: 0x0000
phase 11: 0x0000
phase 12: 0x0000
phase 13: 0x0000
phase 14: 0x0000
phase 15: 0x0000


In [20]:
CTRL    = 0x00
STATUS  = 0x10
RX_DATA = 0x0C
PHASE_SEL  = 0x04

def read_once():
    mmio.write(CTRL, 0x1)          # start_pulse
#     while mmio.read(STATUS) & 1:   # 等 busy=0
#         pass
    return mmio.read(RX_DATA) & 0xFFFF

results = {}
mmio.write(PHASE_SEL, 0x01)
for i in range(16):               
    results[i] = read_once()
    
for i,val in results.items():
    print(f"data {i:2d}: 0x{val:04X}")

data  0: 0x004E
data  1: 0x0054
data  2: 0x004E
data  3: 0x0049
data  4: 0x0054
data  5: 0x0041
data  6: 0x004E
data  7: 0x0049
data  8: 0x004E
data  9: 0x0054
data 10: 0x0041
data 11: 0x0049
data 12: 0x004E
data 13: 0x0054
data 14: 0x0041
data 15: 0x004E


In [10]:
import time

CTRL       = 0x00
STATUS     = 0x10
RX_DATA    = 0x0C
PHASE_SEL  = 0x04

def count_reads_in(mmio, duration_s=1.0, phase=0x01, use_busy=0x00, busy_mask=0x1, busy_timeout_s=0.01):
    """Trigger conversions as fast as possible for `duration_s` and return (count, elapsed_s, rate_hz)."""
    mmio.write(PHASE_SEL, phase)

    read  = mmio.read   # micro-optimization
    write = mmio.write

    t0      = time.monotonic()
    end_t   = t0 + duration_s
    count   = 0

    while time.monotonic() < end_t:
        # start pulse
        write(CTRL, 0x1)
        # 若你的硬體需要 0->1 脈衝，取消下一行註解：
        # write(CTRL, 0x0)

        ok = True
        if use_busy:
            t_wait0 = time.monotonic()
            # 等 busy 清掉；避免卡死，加個超時
            while read(STATUS) & busy_mask:
                if time.monotonic() - t_wait0 > busy_timeout_s:
                    ok = False
                    break

        # 讀資料（即便不使用也要讀，通常會清旗標或彈出 FIFO）
        _ = read(RX_DATA) & 0xFFFF

        if ok:
            count += 1

    elapsed = time.monotonic() - t0
    rate    = count / elapsed if elapsed > 0 else 0.0
    return count, elapsed, rate

# 使用範例（不會列印）：
# cnt, elapsed, rate = count_reads_in(mmio, duration_s=1.0, phase=0x01)

In [18]:
cnt, elapsed, rate = count_reads_in(mmio, duration_s=0.1, phase=0x01)
print(f"{cnt} samples in {elapsed:.3f}s  => {rate:.1f} Hz")

6037 samples in 0.100s  => 60365.8 Hz
